# 2a단계: 추적기 튜닝 (튜닝 클립 0143 · 0157, 정답 미사용)

1b에서 고른 탐지기(yolo26x · imgsz 1920) 위에서 ByteTrack 설정을 고릅니다.
결과는 `tracker_config_v4.json`으로 저장되고, 다음 노트북(step2_v4)이 이 설정을 씁니다.

**실행 순서:** 위에서부터 끝까지. ①의 ROI 그림만 눈으로 확인하면 됩니다.

**런타임: T4 GPU** · 전체 약 15~20분

## 공통 설정

In [ ]:
!pip install -q ultralytics
import os, json, glob, shutil, zipfile, re
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===== 경로 (v3 노트북 Cell 2와 동일하게) =====
FOLDER_PATH      = "/content/drive/MyDrive/2025년 홍익 vs 국민 드론뷰"
DRIVE_OUTPUT_DIR = os.path.join(FOLDER_PATH, "pipeline_outputs")   # v3의 tracking_final.json 위치
EVAL_DIR         = os.path.join(DRIVE_OUTPUT_DIR, "eval")           # 이번에 새로 만드는 평가 폴더
IMG_DIR          = os.path.join(EVAL_DIR, "images")                 # 평가용 프레임 20장 (정답 라벨의 기준 이미지)
PRE_DIR          = os.path.join(EVAL_DIR, "prelabels")              # 사전 라벨
for _d in (EVAL_DIR, IMG_DIR, PRE_DIR):
    os.makedirs(_d, exist_ok=True)

# ===== 클래스 =====
# 정답(GT)에는 0~2만 남아야 합니다. 3(Unknown)은 "라벨러가 확인해야 할 박스" 표시용입니다.
CLASS_NAMES = ["Hongik", "Kookmin", "Referee", "Unknown"]
CLS_ID = {n: i for i, n in enumerate(CLASS_NAMES)}

# ===== 평가 클립 =====
# play_range : 득점 플레이 구간 (v3 frame_id 기준, 1부터 시작, 양 끝 포함)
# scorer     : v3에서 득점 선수 경로를 만들 때 쓴 (track_id, 프레임 구간). 득점 선수 '정답 위치'로 사용.
#              ⚠️ 0156의 #171, #183 구간은 v3 노트북 주석에 '추정, 확인 필요'로 되어 있었습니다.
#                 v3의 프레임 브라우저 셀로 확인 후 틀렸다면 여기 숫자를 고치세요.
EVAL_CLIPS = {
    "DJI_20250913133612_0144_D": {
        "clip_key": 34,
        "play_range": (413, 649),
        "scorer": [(2122, (413, 649))],
    },
    "DJI_20250913135020_0156_D": {
        "clip_key": 44,
        "play_range": (167, 525),
        "scorer": [(184, (167, 214)), (171, (215, 313)), (183, (314, 393)), (208, (410, 525))],
    },
}
FRAMES_PER_CLIP = 10

# ===== 매칭 기준 =====
# 선수가 10~25px로 작아서 IoU 0.5는 박스가 조금만 어긋나도 오답이 됩니다.
# 대신 '박스 중심 거리'로 짝을 짓습니다: 거리 <= max(MIN_PX, RATIO × 정답 박스 긴 변)
MATCH_MIN_PX = 10
MATCH_RATIO  = 0.5
SCORER_MATCH_PX = 15   # 득점 선수 정답 위치(발)와 예측 박스 발 위치가 이 거리(px) 이내면 '찾았다'


def video_path_of(stem):
    for ext in (".MP4", ".mp4", ".MOV", ".mov"):
        p = os.path.join(FOLDER_PATH, stem + ext)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"영상을 찾을 수 없습니다: {stem}")

def tracking_json_of(stem, folder=DRIVE_OUTPUT_DIR):
    return os.path.join(folder, f"{stem}_tracking_final.json")

for stem in EVAL_CLIPS:
    print(stem, "| 영상:", os.path.exists(video_path_of(stem)), "| v3 JSON:", os.path.exists(tracking_json_of(stem)))

In [ ]:
# ===== 공통 함수 =====

def load_tracking(json_path):
    """v3 형식 JSON → {frame_id: [player dict, ...]}, field_polygon"""
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    by_frame = {fr["frame_id"]: fr["players"] for fr in data["frames"]}
    poly = np.array(data.get("field_polygon", []), dtype=np.int32)
    return by_frame, poly


def read_frames_sequential(video_path, frame_ids):
    """
    frame_id(1부터 시작)에 해당하는 프레임을 '처음부터 순서대로 읽어서' 가져옵니다.
    cap.set(POS_FRAMES)로 바로 점프하면 드론 영상(H.264/H.265)에서 몇 프레임씩 어긋날 수 있어서,
    v3가 프레임을 센 방식(처음부터 1, 2, 3...)과 똑같이 맞추기 위해 순차로 읽습니다.
    """
    wanted = set(frame_ids)
    last = max(wanted)
    out = {}
    cap = cv2.VideoCapture(video_path)
    fid = 0
    while fid < last:
        ok, frame = cap.read()
        if not ok:
            break
        fid += 1
        if fid in wanted:
            out[fid] = frame
    cap.release()
    missing = wanted - set(out)
    if missing:
        print(f"⚠️ 읽지 못한 프레임: {sorted(missing)}")
    return out


def pick_frames(play_range, n):
    a, b = play_range
    return sorted(set(np.linspace(a, b, n).round().astype(int).tolist()))


def inside(poly, x, y):
    if poly is None or len(poly) < 3:
        return True
    return cv2.pointPolygonTest(poly.reshape(-1, 1, 2), (float(x), float(y)), False) >= 0


def xyxy_to_yolo_line(cls_id, box, W, H):
    x1, y1, x2, y2 = box
    x1, x2 = max(0, x1), min(W, x2)
    y1, y2 = max(0, y1), min(H, y2)
    cx, cy = (x1 + x2) / 2 / W, (y1 + y2) / 2 / H
    w, h = (x2 - x1) / W, (y2 - y1) / H
    return f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"


def yolo_line_to_xyxy(line, W, H):
    c, cx, cy, w, h = line.split()[:5]
    cx, cy, w, h = float(cx) * W, float(cy) * H, float(w) * W, float(h) * H
    return int(c), [cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2]


def center(b):
    return ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)


def box_iou(a, b):
    """Day 2에서 만든 IoU (xyxy)"""
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def match_by_center(gt_boxes, pred_boxes):
    """
    정답 박스와 예측 박스를 1:1로 짝짓기 (헝가리안 알고리즘, 중심 거리 기준).
    return: [(gi, pi), ...], 짝 없는 정답 인덱스, 짝 없는 예측 인덱스
    """
    G, P = len(gt_boxes), len(pred_boxes)
    if G == 0 or P == 0:
        return [], list(range(G)), list(range(P))
    gc = np.array([center(b) for b in gt_boxes])
    pc = np.array([center(b) for b in pred_boxes])
    dist = np.linalg.norm(gc[:, None, :] - pc[None, :, :], axis=2)          # (G, P)
    gsize = np.array([max(b[2] - b[0], b[3] - b[1]) for b in gt_boxes])
    thr = np.maximum(MATCH_MIN_PX, MATCH_RATIO * gsize)[:, None]            # 정답 박스별 허용 거리
    cost = np.where(dist <= thr, dist, 1e6)
    gi, pi = linear_sum_assignment(cost)
    pairs = [(g, p) for g, p in zip(gi, pi) if cost[g, p] < 1e6]
    mg = {g for g, _ in pairs}
    mp = {p for _, p in pairs}
    return pairs, [g for g in range(G) if g not in mg], [p for p in range(P) if p not in mp]


print("✅ 공통 함수 준비 완료")

## ByteTrack
v3는 Ultralytics 안의 ByteTrack을 썼어요. 여기서는 저장해둔 탐지 결과에 설정만 바꿔 여러 번 돌리려고,
같은 규칙을 numpy로 옮긴 버전을 씁니다. 셀 끝의 자체 점검이 통과하는지 확인하세요.

In [ ]:
# ===== ByteTrack (논문 · Ultralytics 구현과 같은 규칙을 그대로 옮긴 버전) =====
# 저장해둔 탐지 결과에 바로 돌릴 수 있도록 numpy만으로 작성했습니다.
#   1) 칼만 필터로 기존 트랙의 이번 위치 예측
#   2) 높은 점수 박스와 IoU 짝짓기 (헝가리안)
#   3) 짝 못 찾은 트랙은 낮은 점수 박스와 한 번 더 짝짓기  ← ByteTrack의 핵심
#   4) 남은 높은 점수 박스 중 new_track_thresh 이상만 새 트랙으로 등록
#   5) track_buffer 프레임 동안 못 찾은 트랙은 삭제

class KalmanXYAH:
    """상태: [cx, cy, 종횡비 a, 높이 h, 각 속도 4개]. 등속 모델."""
    def __init__(self):
        self.F = np.eye(8)
        for i in range(4):
            self.F[i, 4 + i] = 1.0
        self.H = np.eye(4, 8)
        self.wp, self.wv = 1.0 / 20, 1.0 / 160

    def initiate(self, z):
        h = z[3]
        mean = np.r_[z, np.zeros(4)]
        std = [2 * self.wp * h, 2 * self.wp * h, 1e-2, 2 * self.wp * h,
               10 * self.wv * h, 10 * self.wv * h, 1e-5, 10 * self.wv * h]
        return mean, np.diag(np.square(std))

    def predict(self, mean, cov):
        h = mean[3]
        q = [self.wp * h, self.wp * h, 1e-2, self.wp * h, self.wv * h, self.wv * h, 1e-5, self.wv * h]
        return self.F @ mean, self.F @ cov @ self.F.T + np.diag(np.square(q))

    def update(self, mean, cov, z):
        h = mean[3]
        R = np.diag(np.square([self.wp * h, self.wp * h, 1e-1, self.wp * h]))
        S = self.H @ cov @ self.H.T + R
        K = np.linalg.solve(S, (cov @ self.H.T).T).T
        mean = mean + K @ (z - self.H @ mean)
        cov = cov - K @ S @ K.T
        return mean, cov


def xyxy_to_xyah(b):
    w, h = b[2] - b[0], b[3] - b[1]
    return np.array([(b[0] + b[2]) / 2, (b[1] + b[3]) / 2, w / max(h, 1e-6), h], float)


def xyah_to_xyxy(m):
    cx, cy, a, h = m[:4]
    w = a * h
    return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], float)


def iou_matrix_np(a, b):
    a = np.asarray(a, float).reshape(-1, 4); b = np.asarray(b, float).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None); inter = wh[..., 0] * wh[..., 1]
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def linear_assign(cost, thresh):
    if cost.size == 0:
        return [], list(range(cost.shape[0])), list(range(cost.shape[1]))
    c = np.where(cost <= thresh, cost, 1e6)
    r, k = linear_sum_assignment(c)
    m = [(i, j) for i, j in zip(r, k) if c[i, j] <= thresh]
    mi = {i for i, _ in m}; mj = {j for _, j in m}
    return m, [i for i in range(cost.shape[0]) if i not in mi], [j for j in range(cost.shape[1]) if j not in mj]


class Track:
    _next_id = 0
    TRACKED, LOST, REMOVED = 0, 1, 2

    def __init__(self, box, score, det_idx, kf):
        self.kf, self.box, self.score, self.det_idx = kf, np.asarray(box, float), score, det_idx
        self.mean = self.cov = None
        self.state, self.activated, self.track_id = Track.TRACKED, False, -1
        self.start_frame = self.end_frame = 0

    def predicted_box(self):
        return xyah_to_xyxy(self.mean) if self.mean is not None else self.box

    def predict(self):
        m = self.mean.copy()
        if self.state != Track.TRACKED:
            m[7] = 0.0                          # 놓친 트랙은 높이 변화 속도 0
        self.mean, self.cov = self.kf.predict(m, self.cov)

    def activate(self, frame_id):
        Track._next_id += 1
        self.track_id = Track._next_id
        self.mean, self.cov = self.kf.initiate(xyxy_to_xyah(self.box))
        self.state = Track.TRACKED
        self.activated = frame_id == 1
        self.start_frame = self.end_frame = frame_id

    def update(self, det, frame_id):
        self.mean, self.cov = self.kf.update(self.mean, self.cov, xyxy_to_xyah(det.box))
        self.box, self.score, self.det_idx = det.box, det.score, det.det_idx
        self.state, self.activated, self.end_frame = Track.TRACKED, True, frame_id


class ByteTracker:
    def __init__(self, track_high_thresh=0.5, track_low_thresh=0.1, new_track_thresh=0.6,
                 track_buffer=30, match_thresh=0.8, fuse_score=True, frame_rate=30):
        self.hi, self.lo, self.new = track_high_thresh, track_low_thresh, new_track_thresh
        self.max_lost = int(frame_rate / 30.0 * track_buffer)
        self.match, self.fuse = match_thresh, fuse_score
        self.kf = KalmanXYAH()
        self.tracked, self.lost = [], []
        self.frame_id = 0
        Track._next_id = 0

    def _dist(self, tracks, dets, fuse):
        d = 1 - iou_matrix_np([t.predicted_box() for t in tracks], [x.box for x in dets])
        if fuse and len(dets):
            d = 1 - (1 - d) * np.array([x.score for x in dets])[None, :]
        return d

    def update(self, boxes, scores):
        """boxes (N,4) xyxy, scores (N,) → [(track_id, det_idx), ...] 이번 프레임에 확정된 트랙"""
        self.frame_id += 1
        f = self.frame_id
        boxes = np.asarray(boxes, float).reshape(-1, 4); scores = np.asarray(scores, float).reshape(-1)
        high = [Track(boxes[i], scores[i], i, self.kf) for i in np.where(scores >= self.hi)[0]]
        low = [Track(boxes[i], scores[i], i, self.kf) for i in np.where((scores > self.lo) & (scores < self.hi))[0]]

        unconfirmed = [t for t in self.tracked if not t.activated]
        confirmed = [t for t in self.tracked if t.activated]
        pool = confirmed + [t for t in self.lost if t not in confirmed]
        for t in pool:
            t.predict()

        refound, activated_now = [], []
        # 1차: 높은 점수 박스
        m, u_t, u_d = linear_assign(self._dist(pool, high, self.fuse), self.match)
        for i, j in m:
            t = pool[i]
            if t.state != Track.TRACKED:
                refound.append(t)
            t.update(high[j], f); activated_now.append(t)
        # 2차: 남은 '추적 중' 트랙 × 낮은 점수 박스
        r_tracked = [pool[i] for i in u_t if pool[i].state == Track.TRACKED]
        m2, u_t2, _ = linear_assign(self._dist(r_tracked, low, False), 0.5)
        for i, j in m2:
            r_tracked[i].update(low[j], f); activated_now.append(r_tracked[i])
        for i in u_t2:
            r_tracked[i].state = Track.LOST
        # 확정 안 된(1프레임짜리) 트랙 × 남은 높은 점수 박스
        rest_high = [high[j] for j in u_d]
        m3, u_un, u_d3 = linear_assign(self._dist(unconfirmed, rest_high, self.fuse), 0.7)
        for i, j in m3:
            unconfirmed[i].update(rest_high[j], f); activated_now.append(unconfirmed[i])
        for i in u_un:
            unconfirmed[i].state = Track.REMOVED
        # 새 트랙
        new_tracks = []
        for j in u_d3:
            d = rest_high[j]
            if d.score >= self.new:
                d.activate(f); new_tracks.append(d)
        # 오래 못 찾은 트랙 삭제
        for t in pool:
            if t.state == Track.LOST and f - t.end_frame > self.max_lost:
                t.state = Track.REMOVED

        everyone = {id(t): t for t in self.tracked + self.lost + new_tracks}
        self.tracked = [t for t in everyone.values() if t.state == Track.TRACKED]
        self.lost = [t for t in everyone.values() if t.state == Track.LOST]
        return [(t.track_id, t.det_idx) for t in self.tracked if t.activated and t.end_frame == f]


def _bytetrack_selftest():
    """작은 합성 장면으로 동작 확인: 점수가 잠깐 낮아지거나 몇 프레임 사라져도 ID 유지, 약한 새 박스는 새 ID 안 받음"""
    tr = ByteTracker(track_high_thresh=0.5, track_low_thresh=0.1, new_track_thresh=0.6, track_buffer=30)
    ids = defaultdict(set)
    for f in range(1, 61):
        boxes, scores = [], []
        x = 100 + 3 * f                                     # 선수 A: 오른쪽으로 이동
        if not (30 <= f < 36):                              # 30~35프레임 동안 사라짐(가려짐)
            boxes.append([x, 100, x + 12, 130]); scores.append(0.3 if 15 <= f < 20 else 0.8)   # 15~19 낮은 점수
        boxes.append([300, 200 + f, 312, 230 + f]); scores.append(0.9)                      # 선수 B
        if f == 40:
            boxes.append([500, 50, 512, 80]); scores.append(0.4)                            # 약한 오탐 1번
        for tid, di in tr.update(boxes, scores):
            ids["A" if boxes[di][0] < 290 else ("B" if boxes[di][0] < 400 else "X")].add(tid)
    assert len(ids["A"]) == 1 and len(ids["B"]) == 1 and "X" not in ids, dict(ids)
    print("✅ ByteTrack 자체 점검 통과:", {k: sorted(v) for k, v in ids.items()})

_bytetrack_selftest()

---
# 2a. 추적기 튜닝 — 평가 클립이 아닌 클립에서, 정답 없이

| 순서 | 내용 | 시간 (T4) |
|---|---|---|
| ① | 튜닝 클립 확인 + 평가 클립의 1a ROI를 화면 정렬로 옮겨오기 | 1분 |
| ② | yolo26x(1920)로 튜닝 클립 전체 탐지 → Drive에 저장 (한 번만) | 약 10분 |
| ③ | ByteTrack 설정 55가지를 저장된 탐지 결과에 돌리기 | 2~5분 |
| ④ | **미리 정한 규칙**으로 설정 선택 → `tracker_config_v4.json` 저장 | 즉시 |

**이 노트북은 정답 라벨(gt_roboflow.zip)을 불러오지 않습니다.** 평가셋에 맞춘 설정이 되는 것을 막기 위해서예요.

**런타임: T4 GPU**

In [ ]:
import time, pickle
from ultralytics import YOLO

# 튜닝 클립 → 같은 드론 위치에서 찍힌 평가 클립 (ROI를 옮겨올 기준)
TUNE_CLIPS = {
    "DJI_20250913133530_0143_D": "DJI_20250913133612_0144_D",   # 0144보다 2분 앞
    "DJI_20250913135111_0157_D": "DJI_20250913135020_0156_D",   # 0156보다 1분 뒤
}
DET_WEIGHTS, DET_IMGSZ, DET_CONF_CACHE = "yolo26x.pt", 1920, 0.05
CACHE_DIR = os.path.join(EVAL_DIR, "cache"); os.makedirs(CACHE_DIR, exist_ok=True)
TRACKER_CFG_PATH = os.path.join(EVAL_DIR, "tracker_config_v4.json")

ROI_1A = {}
for ev in set(TUNE_CLIPS.values()):
    p = os.path.join(EVAL_DIR, "runs", "step1a_roi", f"{ev}_tracking_final.json")
    assert os.path.exists(p), f"1a 결과가 없습니다: {p}"
    _, ROI_1A[ev] = load_tracking(p)
for stem in TUNE_CLIPS:
    print(stem, "영상:", os.path.exists(video_path_of(stem)))


def first_frame(stem):
    cap = cv2.VideoCapture(video_path_of(stem)); ok, img = cap.read(); cap.release()
    assert ok, f"{stem} 첫 프레임을 읽지 못했습니다"
    return img


def eval_reference_image(ev):
    """평가 클립의 기준 이미지: 0단계에서 저장한 평가 프레임 중 첫 장"""
    f = sorted(n for n in os.listdir(IMG_DIR) if n.startswith(ev) and n.endswith(".jpg"))[0]
    return cv2.imread(os.path.join(IMG_DIR, f))

In [ ]:
# ① 평가 클립 ROI를 튜닝 클립으로 옮겨오기
#    배경(라인, 골대, 펜스)은 그대로이고 드론만 조금 움직였다고 보고, 두 화면을 정렬하는 변환을 찾아 ROI에 적용
def align_affine(src_img, dst_img):
    """src 화면 좌표 → dst 화면 좌표로 보내는 2x3 변환. ORB 특징점 매칭 + RANSAC. (정렬 품질 = inlier 비율)"""
    g1 = cv2.cvtColor(src_img, cv2.COLOR_BGR2GRAY); g2 = cv2.cvtColor(dst_img, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(5000)
    k1, d1 = orb.detectAndCompute(g1, None); k2, d2 = orb.detectAndCompute(g2, None)
    if d1 is None or d2 is None or len(k1) < 20 or len(k2) < 20:
        return None, 0.0
    matches = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True).match(d1, d2)
    if len(matches) < 20:
        return None, 0.0
    p1 = np.float32([k1[m.queryIdx].pt for m in matches]); p2 = np.float32([k2[m.trainIdx].pt for m in matches])
    M, inl = cv2.estimateAffinePartial2D(p1, p2, method=cv2.RANSAC, ransacReprojThreshold=3.0)
    if M is None:
        return None, 0.0
    return M, float(inl.mean())


TUNE_ROI, ALIGN_INFO = {}, {}
for stem, ev in TUNE_CLIPS.items():
    src, dst = eval_reference_image(ev), first_frame(stem)
    M, q = align_affine(src, dst)
    if M is not None and q >= 0.3:
        poly = cv2.transform(ROI_1A[ev].reshape(-1, 1, 2).astype(np.float32), M).reshape(-1, 2)
        H, W = dst.shape[:2]
        poly[:, 0] = np.clip(poly[:, 0], 0, W - 1); poly[:, 1] = np.clip(poly[:, 1], 0, H - 1)
        TUNE_ROI[stem] = poly.round().astype(np.int32)
        ALIGN_INFO[stem] = {"방법": "평가 클립 ROI를 정렬로 이전", "정렬 품질(inlier)": round(q, 2),
                            "이동(px)": [round(float(M[0, 2]), 1), round(float(M[1, 2]), 1)],
                            "배율": round(float(np.hypot(M[0, 0], M[1, 0])), 3)}
    else:
        rp = os.path.join(DRIVE_OUTPUT_DIR, "roi_overrides.json")
        ov = json.load(open(rp, encoding="utf-8")) if os.path.exists(rp) else {}
        key = os.path.basename(video_path_of(stem))
        assert key in ov, f"{stem}: 정렬 실패 + v3 ROI도 없음 → 이 클립 대신 다른 튜닝 클립을 고르세요"
        TUNE_ROI[stem] = np.array(ov[key], np.int32)
        ALIGN_INFO[stem] = {"방법": "⚠️ 정렬 실패 → v3 roi_overrides 사용 (그림으로 꼭 확인)", "정렬 품질(inlier)": round(q, 2)}
    print(stem, ALIGN_INFO[stem])

fig, axes = plt.subplots(len(TUNE_CLIPS), 2, figsize=(18, 5.2 * len(TUNE_CLIPS)))
axes = np.atleast_2d(axes)
for row, (stem, ev) in zip(axes, TUNE_CLIPS.items()):
    a = eval_reference_image(ev).copy(); cv2.polylines(a, [ROI_1A[ev].reshape(-1, 1, 2)], True, (0, 255, 0), 3)
    b = first_frame(stem).copy(); cv2.polylines(b, [TUNE_ROI[stem].reshape(-1, 1, 2)], True, (0, 255, 255), 3)
    row[0].imshow(cv2.cvtColor(a, cv2.COLOR_BGR2RGB)); row[0].set_title(f"평가 클립 {ev[-6:-2]} — 1a ROI"); row[0].axis("off")
    row[1].imshow(cv2.cvtColor(b, cv2.COLOR_BGR2RGB)); row[1].set_title(f"튜닝 클립 {stem[-6:-2]} — 옮겨온 ROI"); row[1].axis("off")
plt.tight_layout(); plt.show()
print("👉 오른쪽 노란 선이 경기장 라인을 잘 따라가는지 확인하세요. 크게 어긋나면 TUNE_CLIPS에서 다른 클립을 고르세요.")

In [ ]:
# ② 튜닝 클립 전체 탐지 → 캐시 (팀 분류 특징은 필요 없음)
def tune_cache_path(stem):
    return os.path.join(CACHE_DIR, f"{stem}_tune_y26x{DET_IMGSZ}_conf{DET_CONF_CACHE}.pkl")


def build_tune_cache(stem, model):
    local = os.path.join("/content/_scratch_video", os.path.basename(video_path_of(stem)))
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if not os.path.exists(local):
        shutil.copy(video_path_of(stem), local)
    cap = cv2.VideoCapture(local)
    frames, fid, t0 = {}, 0, time.time()
    while True:
        ok, img = cap.read()
        if not ok:
            break
        fid += 1
        r = model.predict(img, imgsz=DET_IMGSZ, conf=DET_CONF_CACHE, classes=[0], max_det=300, verbose=False)[0]
        bx = r.boxes.xyxy.cpu().numpy().reshape(-1, 4); sc = r.boxes.conf.cpu().numpy().reshape(-1)
        keep = [i for i, bb in enumerate(bx) if inside(TUNE_ROI[stem], (bb[0] + bb[2]) / 2, bb[3])]
        frames[fid] = {"boxes": bx[keep], "scores": sc[keep]}
        if fid % 200 == 0:
            print(f"  [{stem}] {fid}프레임 ({time.time() - t0:.0f}s)")
    cap.release()
    data = {"stem": stem, "start": 1, "end": fid, "frames": frames, "roi": TUNE_ROI[stem].tolist()}
    with open(tune_cache_path(stem), "wb") as f:
        pickle.dump(data, f)
    print(f"✅ {stem}: {fid}프레임 캐시 저장 ({time.time() - t0:.0f}s)")
    return data


TUNE_CACHE, _model = {}, None
for stem in TUNE_CLIPS:
    if os.path.exists(tune_cache_path(stem)):
        TUNE_CACHE[stem] = pickle.load(open(tune_cache_path(stem), "rb"))
        print(f"⏭️ {stem}: 저장된 캐시 사용 ({TUNE_CACHE[stem]['end']}프레임)")
        continue
    if _model is None:
        _model = YOLO(DET_WEIGHTS)
    TUNE_CACHE[stem] = build_tune_cache(stem, _model)

## ③~④ 설정 비교와 선택 규칙 (결과를 보기 전에 정한 규칙)

정답이 없으니, 추적 결과만으로 계산할 수 있는 세 지표를 씁니다.

| 지표 | 뜻 | 좋은 방향 |
|---|---|---|
| **커버율** | 추적 결과로 남은 박스 수 ÷ 확실한 탐지(점수 0.5 이상) 수 | 높을수록 (선수를 덜 버림, Recall 대용) |
| **ID 쪼개짐** | 고유 ID 수 ÷ 프레임당 평균 추적 인원 | 낮을수록 (한 사람이 여러 ID로 쪼개지지 않음) |
| **짧은 트랙 비율** | 15프레임(0.5초) 미만으로 끝난 ID의 비율 | 낮을수록 (잡음 트랙·끊김이 적음) |

**선택 규칙:** ID 쪼개짐과 짧은 트랙 비율이 모두 **v3 설정보다 10% 넘게 나빠지지 않는** 설정 중에서 **커버율이 가장 높은** 것.
커버율이 같으면(0.005 이내) ID 쪼개짐이 낮은 쪽. 조건을 만족하는 설정이 없으면 v3 설정을 유지합니다.

In [ ]:
import itertools
V3_CFG = dict(track_high_thresh=0.5, track_low_thresh=0.15, new_track_thresh=0.6, track_buffer=40, match_thresh=0.85)
GRID = [dict(track_high_thresh=h, track_low_thresh=0.1, new_track_thresh=n, track_buffer=b, match_thresh=m)
        for h, n, b, m in itertools.product([0.25, 0.35, 0.5], [0.25, 0.4, 0.6], [30, 60, 90], [0.8, 0.9])]
CONFIGS = {"v3설정": V3_CFG, **{f"h{c['track_high_thresh']}_n{c['new_track_thresh']}_b{c['track_buffer']}_m{c['match_thresh']}": c for c in GRID}}
SHORT_LEN = 15


def tune_metrics(params):
    """클립별로 계산한 뒤 프레임 수로 가중 평균 (ID 쪼개짐 = 그 클립의 고유 ID ÷ 그 클립의 프레임당 평균 추적 인원)"""
    out_boxes = conf_boxes = 0
    frags, weights, lengths, people = [], [], [], []
    for stem, c in TUNE_CACHE.items():
        tr = ByteTracker(**params, fuse_score=True, frame_rate=30)
        life, counts = Counter(), []
        for fid in range(c["start"], c["end"] + 1):
            fr = c["frames"].get(fid, {"boxes": np.zeros((0, 4)), "scores": np.zeros(0)})
            res = tr.update(fr["boxes"], fr["scores"])
            out_boxes += len(res); conf_boxes += int((fr["scores"] >= 0.5).sum())
            counts.append(len(res))
            for tid, _ in res:
                life[tid] += 1
        avg = float(np.mean(counts)) if counts else 0.0
        frags.append(len(life) / max(avg, 1e-6)); weights.append(len(counts))
        lengths += list(life.values()); people.append(avg)
    return {"커버율": round(out_boxes / max(conf_boxes, 1), 4),
            "ID 쪼개짐": round(float(np.average(frags, weights=weights)), 3),
            "짧은 트랙 비율": round(float(np.mean([l < SHORT_LEN for l in lengths])) if lengths else 0.0, 3),
            "프레임당 추적 인원": round(float(np.average(people, weights=weights)), 1), "고유 ID 수": len(lengths)}


t0, rows = time.time(), []
for i, (name, params) in enumerate(CONFIGS.items(), 1):
    rows.append({"설정": name, **tune_metrics(params), **params})
    if i % 10 == 0:
        print(f"  {i}/{len(CONFIGS)} 설정 완료 ({time.time() - t0:.0f}s)")
tune_df = pd.DataFrame(rows).set_index("설정")
print(f"✅ {len(CONFIGS)}개 설정 비교 완료 ({time.time() - t0:.0f}s)")

In [ ]:
base = tune_df.loc["v3설정"]
ok = tune_df[(tune_df["ID 쪼개짐"] <= base["ID 쪼개짐"] * 1.10) &
             (tune_df["짧은 트랙 비율"] <= base["짧은 트랙 비율"] * 1.10 + 1e-9)]
if len(ok):
    top = ok["커버율"].max()
    CHOSEN = ok[ok["커버율"] >= top - 0.005].sort_values("ID 쪼개짐").index[0]
else:
    CHOSEN = "v3설정"

show = ["커버율", "ID 쪼개짐", "짧은 트랙 비율", "프레임당 추적 인원", "고유 ID 수"]
print("=== v3 설정 ==="); display(tune_df.loc[["v3설정"], show])
print(f"=== 조건을 만족한 설정 {len(ok)}개 중 커버율 상위 10개 ==="); display(ok.sort_values("커버율", ascending=False)[show].head(10))
print(f"\n✅ 선택: {CHOSEN}\n   {CONFIGS[CHOSEN]}")
print(f"   커버율 {base['커버율']} → {tune_df.loc[CHOSEN, '커버율']}, "
      f"ID 쪼개짐 {base['ID 쪼개짐']} → {tune_df.loc[CHOSEN, 'ID 쪼개짐']}, "
      f"짧은 트랙 {base['짧은 트랙 비율']} → {tune_df.loc[CHOSEN, '짧은 트랙 비율']}")

fig, ax = plt.subplots(figsize=(9, 6))
sc_ = ax.scatter(tune_df["ID 쪼개짐"], tune_df["커버율"], c=tune_df["짧은 트랙 비율"], cmap="viridis", s=35)
ax.scatter([base["ID 쪼개짐"]], [base["커버율"]], marker="s", s=150, facecolors="none", edgecolors="red", lw=2, label="v3 설정")
ax.scatter([tune_df.loc[CHOSEN, "ID 쪼개짐"]], [tune_df.loc[CHOSEN, "커버율"]], marker="*", s=300, c="red", label=f"선택: {CHOSEN}")
ax.axvline(base["ID 쪼개짐"] * 1.10, color="gray", ls="--", lw=1)
plt.colorbar(sc_, label="짧은 트랙 비율"); ax.set_xlabel("ID 쪼개짐 (낮을수록 좋음)"); ax.set_ylabel("커버율 (높을수록 좋음)")
ax.set_title("튜닝 클립에서의 추적기 설정 비교 (왼쪽 위가 좋음, 회색 점선 = 허용 한계)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 선택한 설정 저장 → v4 노트북이 이 파일을 읽어서 씁니다
record_cfg = {
    "chosen": CHOSEN, "params": CONFIGS[CHOSEN], "v3_params": V3_CFG,
    "tune_clips": list(TUNE_CLIPS), "detector": f"{DET_WEIGHTS} imgsz{DET_IMGSZ}",
    "rule": "ID 쪼개짐·짧은 트랙 비율이 v3설정 대비 10% 이내 악화인 설정 중 커버율 최대 (동률 시 쪼개짐 낮은 쪽)",
    "metrics_chosen": tune_df.loc[CHOSEN, show].to_dict(), "metrics_v3": tune_df.loc["v3설정", show].to_dict(),
    "roi_alignment": ALIGN_INFO, "n_configs": len(CONFIGS),
}
with open(TRACKER_CFG_PATH, "w", encoding="utf-8") as f:
    json.dump(record_cfg, f, ensure_ascii=False, indent=2, default=float)
tune_df.to_csv(os.path.join(EVAL_DIR, "tracker_tuning_results.csv"), encoding="utf-8-sig")
print("✅ 저장:", TRACKER_CFG_PATH)
print("   다음: step2_v4 노트북을 실행하면 이 설정으로 v4를 만들고 평가셋으로 한 번 채점합니다.")